In [1]:
import pandas as pd


# LOAD ONLY REQUIRED COLUMNS

orders = pd.read_csv(
"orders.csv",
usecols=["order_id", "user_id"]
)

order_products = pd.read_csv(
"order_products__train.csv",
usecols=["order_id", "product_id"]
)

products = pd.read_csv(
"products.csv",
usecols=["product_id", "product_name"]
)


# SAMPLE DATA (CRITICAL)

orders = orders.sample(n=30000, random_state=42)
order_products = order_products.sample(n=60000, random_state=42)


# CLEAN USERS

users = orders[['user_id']].dropna().drop_duplicates()
users['user_id'] = users['user_id'].astype(int)


# CLEAN PRODUCTS

products_cleaned = products.dropna().drop_duplicates()

# keep only products in sampled interactions
products_cleaned = products_cleaned[
products_cleaned['product_id'].isin(order_products['product_id'])
]

products_cleaned['product_id'] = products_cleaned['product_id'].astype(int)


# CLEAN INTERACTIONS

interactions = order_products.merge(
orders,
on='order_id',
how='inner'
)

interactions = interactions[['user_id', 'product_id']]
interactions.dropna(inplace=True)
interactions.drop_duplicates(inplace=True)

interactions['user_id'] = interactions['user_id'].astype(int)
interactions['product_id'] = interactions['product_id'].astype(int)
interactions['interaction'] = 1


# USER–ITEM MATRIX (SAFE SIZE)

user_item_matrix = interactions.pivot_table(
index='user_id',
columns='product_id',
values='interaction',
aggfunc='sum',
fill_value=0
)


# SAVE OUTPUTS

users.to_csv("cleaned_users.csv", index=False)
products_cleaned.to_csv("cleaned_products.csv", index=False)
interactions.to_csv("cleaned_interactions.csv", index=False)
user_item_matrix.to_csv("user_item_matrix.csv")


# DONE

print("✅ Milestone 1 completed using sampled data")
print("Users:", users.shape)
print("Products:", products_cleaned.shape)
print("Interactions:", interactions.shape)
print("User–Item Matrix:", user_item_matrix.shape)

✅ Milestone 1 completed using sampled data
Users: (26294, 1)
Products: (13338, 2)
Interactions: (492, 3)
User–Item Matrix: (370, 415)
